# BOS Pipeline — End-to-End Demo

This notebook demonstrates the full Background-Oriented Schlieren (BOS) 
processing pipeline using **synthetic data** (no camera hardware required).

Steps covered:
1. Generate synthetic reference + measurement images (Gaussian refractive-index blob)
2. Preprocessing: Gaussian filtering, optional flat-field correction
3. Displacement field computation (cross-correlation, Farneback, Lucas-Kanade)
4. Abel inversion for the axisymmetric geometry
5. Visualisation: magnitude map, quiver plot, Abel field, summary panel
6. Export: HDF5 displacement, JSON processing log

---
**Install requirements first:**
```bash
pip install -e ..[dev]   # from repo root
# or
pip install -r requirements.txt
```

In [ ]:
%matplotlib inline
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('.').resolve().parent))

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Notebook output directory
OUTPUT_DIR = Path('./demo_output')
OUTPUT_DIR.mkdir(exist_ok=True)
print('Output will be saved to:', OUTPUT_DIR.resolve())

## 1. Generate Synthetic Data

In [ ]:
from tests.generate_synthetic_data import SyntheticConfig, generate, save_dataset

# --- Cartesian geometry ---
cfg_cart = SyntheticConfig(
    height=512, width=512,
    blob_sigma_x=40.0, blob_sigma_y=30.0,
    blob_amplitude=0.05,
    sensitivity=50.0,
    noise_std=0.002,
    geometry='cartesian',
    seed=42,
)
ref_cart, meas_cart, dx_true, dy_true, dn_cart = generate(cfg_cart)

# --- Axisymmetric geometry (hydrogen jet) ---
cfg_axi = SyntheticConfig(
    height=512, width=512,
    blob_sigma_x=30.0, blob_sigma_y=50.0,
    blob_amplitude=0.06,
    sensitivity=50.0,
    noise_std=0.002,
    geometry='axisymmetric',
    seed=42,
)
ref_axi, meas_axi, dx_axi, dy_axi, dn_axi = generate(cfg_axi)

print(f'Cartesian:    reference {ref_cart.shape}, max |dx_true|={np.abs(dx_true).max():.2f} px')
print(f'Axisymmetric: reference {ref_axi.shape},  max |dx_axi| ={np.abs(dx_axi).max():.2f} px')

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
titles = ['Reference', 'Measurement', 'dx_true [px]', 'dy_true [px]']
for col, (img, title) in enumerate(zip(
        [ref_cart, meas_cart, dx_true, dy_true], titles)):
    cmap = 'gray' if col < 2 else 'seismic'
    axes[0, col].imshow(img, cmap=cmap, origin='upper')
    axes[0, col].set_title(f'Cartesian — {title}', fontsize=9)
    axes[0, col].axis('off')

for col, (img, title) in enumerate(zip(
        [ref_axi, meas_axi, dx_axi, dy_axi], titles)):
    cmap = 'gray' if col < 2 else 'seismic'
    axes[1, col].imshow(img, cmap=cmap, origin='upper')
    axes[1, col].set_title(f'Axisym — {title}', fontsize=9)
    axes[1, col].axis('off')

plt.suptitle('Synthetic BOS Data', fontsize=12)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'synthetic_data_overview.png', dpi=150)
plt.show()

## 2. Preprocessing

In [ ]:
from bos_pipeline.processing.preprocess import PreprocessConfig, preprocess

pre_cfg = PreprocessConfig(
    gaussian_sigma=1.5,
    reference_avg_frames=1,   # single frames in synthetic case
    measurement_avg_frames=1,
)

ref_p, meas_p = preprocess(ref_cart, meas_cart, config=pre_cfg)
print(f'Preprocessed: dtype={ref_p.dtype}, range=[{ref_p.min():.3f}, {ref_p.max():.3f}]')

## 3. Displacement Field — Cross-Correlation

In [ ]:
from bos_pipeline.processing.displacement import (
    DisplacementConfig, compute_displacement,
    displacement_magnitude, interpolate_to_full_resolution,
)

xcorr_cfg = DisplacementConfig(
    method='cross_correlation',
    window_size=64,
    overlap=0.75,
    subpixel='parabolic',
)

dx_xcorr, dy_xcorr = compute_displacement(ref_p, meas_p, config=xcorr_cfg)
print(f'Grid shape:  {dx_xcorr.shape}')
print(f'max |dx|={np.abs(dx_xcorr).max():.2f} px   max |dy|={np.abs(dy_xcorr).max():.2f} px')

# Up-sample to full resolution for comparison with ground truth
gx = dx_xcorr._grid_x
gy = dx_xcorr._grid_y
dx_full, dy_full = interpolate_to_full_resolution(
    dx_xcorr, dy_xcorr, ref_p.shape, gx, gy
)
mag_full = displacement_magnitude(dx_full, dy_full)
mag_true = displacement_magnitude(dx_true, dy_true)
print(f'\nGround truth max magnitude: {mag_true.max():.2f} px')
print(f'Computed  max magnitude:    {mag_full.max():.2f} px')

In [ ]:
from bos_pipeline import visualization as viz

fig, _ = viz.plot_displacement_magnitude(
    dx_full, dy_full, title='Cross-Correlation — Displacement Magnitude', dpi=100
)
viz.save_figure(fig, OUTPUT_DIR, 'xcorr_magnitude', ['png'])
plt.show()

fig, _ = viz.plot_displacement_components(
    dx_full, dy_full, title='Cross-Correlation — dx / dy', dpi=100
)
viz.save_figure(fig, OUTPUT_DIR, 'xcorr_components', ['png'])
plt.show()

fig, _ = viz.plot_quiver(
    ref_p, dx_full, dy_full, downsample=16,
    title='Cross-Correlation — Quiver', dpi=100
)
viz.save_figure(fig, OUTPUT_DIR, 'xcorr_quiver', ['png'])
plt.show()

## 4. Displacement Field — Farneback Optical Flow

In [ ]:
try:
    import cv2
    fb_cfg = DisplacementConfig(
        method='farneback',
        fb_pyr_scale=0.5, fb_levels=4, fb_winsize=15,
        fb_iterations=5, fb_poly_n=5, fb_poly_sigma=1.2,
    )
    dx_fb, dy_fb = compute_displacement(ref_p, meas_p, config=fb_cfg)
    print(f'Farneback max |dx|={np.abs(dx_fb).max():.2f} px')

    fig, _ = viz.plot_displacement_magnitude(
        dx_fb, dy_fb, title='Farneback — Displacement Magnitude', dpi=100
    )
    viz.save_figure(fig, OUTPUT_DIR, 'farneback_magnitude', ['png'])
    plt.show()
except ImportError:
    print('opencv-python not installed — skipping Farneback demo.')

## 5. Abel Inversion (Axisymmetric Geometry)

In [ ]:
try:
    import abel  # PyAbel
    from bos_pipeline.processing.abel import AbelConfig, abel_invert

    # Process axisymmetric dataset
    ref_ap, meas_ap = preprocess(ref_axi, meas_axi, config=pre_cfg)
    dx_axi_c, dy_axi_c = compute_displacement(ref_ap, meas_ap, config=xcorr_cfg)
    gx_a = dx_axi_c._grid_x
    gy_a = dx_axi_c._grid_y
    dx_axi_full, dy_axi_full = interpolate_to_full_resolution(
        dx_axi_c, dy_axi_c, ref_ap.shape, gx_a, gy_a
    )

    abel_cfg = AbelConfig(
        enabled=True,
        method='three_point',
        component='dx',
        axis_mode='auto',
    )
    inv_field, axis_arr = abel_invert(dx_axi_full, dy_axi_full, config=abel_cfg)
    axis_col = int(axis_arr[0])
    print(f'Abel inversion complete. Symmetry axis at column {axis_col}.')
    print(f'Inverted field range: [{inv_field.min():.4f}, {inv_field.max():.4f}]')

    fig, _ = viz.plot_abel_field(
        inv_field, axis_col=axis_col,
        title='Abel-Inverted Field (three_point)', dpi=100
    )
    viz.save_figure(fig, OUTPUT_DIR, 'abel_field', ['png'])
    plt.show()

except ImportError:
    print('PyAbel not installed — skipping Abel inversion demo.')
    print('Install with: pip install PyAbel')

## 6. Summary Panel

In [ ]:
try:
    inv_f = inv_field
except NameError:
    inv_f = None  # Abel not available

fig, _ = viz.plot_summary(
    ref_p, meas_p, dx_full, dy_full,
    inv_field=inv_f, dpi=100
)
viz.save_figure(fig, OUTPUT_DIR, 'summary_panel', ['png', 'pdf'])
plt.show()

## 7. Side-by-Side Comparison

In [ ]:
fig, _ = viz.plot_side_by_side(
    ref_p, meas_p,
    titles=('Reference (preprocessed)', 'Measurement (preprocessed)'),
    dpi=100,
)
viz.save_figure(fig, OUTPUT_DIR, 'side_by_side', ['png'])
plt.show()

## 8. Export Results

In [ ]:
from bos_pipeline import export

# HDF5
paths_h5 = export.export_displacement(
    dx_full, dy_full,
    output_dir=OUTPUT_DIR,
    stem='demo_displacement',
    fmt='hdf5',
)
print('HDF5 saved:', paths_h5)

# NumPy
paths_npy = export.export_displacement(
    dx_full, dy_full,
    output_dir=OUTPUT_DIR,
    stem='demo_displacement',
    fmt='npy',
)
print('NPY saved:', paths_npy)

In [ ]:
import json

log_path = export.write_log(
    output_dir=OUTPUT_DIR,
    config={
        'preprocessing': {'gaussian_sigma': 1.5},
        'displacement': {'method': 'cross_correlation', 'window_size': 64, 'overlap': 0.75},
    },
    input_paths={'synthetic': 'generated in notebook'},
    output_paths={k: str(v) for k, v in paths_h5.items()},
    processing_stats={
        'max_dx_px': float(np.abs(dx_full).max()),
        'max_dy_px': float(np.abs(dy_full).max()),
        'image_shape': list(ref_p.shape),
    },
    stem='demo_log',
)

with open(log_path) as f:
    log = json.load(f)
print('Processing log (stats):', json.dumps(log['stats'], indent=2))

## 9. Verify HDF5 Round-Trip

In [ ]:
import h5py

h5_file = paths_h5['hdf5']
with h5py.File(str(h5_file), 'r') as f:
    dx_loaded = f['dx'][:]
    dy_loaded = f['dy'][:]
    mag_loaded = f['magnitude'][:]
    print('HDF5 datasets:', list(f.keys()))
    print('Attributes:', dict(f.attrs))

np.testing.assert_allclose(dx_loaded, dx_full, atol=1e-6)
np.testing.assert_allclose(dy_loaded, dy_full, atol=1e-6)
print('Round-trip check passed.')

## 10. Accuracy Check — Cross-Correlation vs Ground Truth

In [ ]:
# Compare computed displacement magnitude with ground truth
mag_computed = displacement_magnitude(dx_full, dy_full)
mag_gt = displacement_magnitude(dx_true, dy_true)

# Only compare in the central region (away from edges where interpolation degrades)
pad = 32
H, W = mag_computed.shape
crop_computed = mag_computed[pad:H-pad, pad:W-pad]
crop_gt = mag_gt[pad:H-pad, pad:W-pad]

rmse = np.sqrt(np.mean((crop_computed - crop_gt) ** 2))
mae = np.abs(crop_computed - crop_gt).mean()
max_gt = crop_gt.max()

print(f'Displacement magnitude comparison (central crop):')
print(f'  Ground truth max:  {max_gt:.3f} px')
print(f'  RMSE:              {rmse:.3f} px  ({rmse/max_gt*100:.1f}% of max)')
print(f'  MAE:               {mae:.3f} px')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
vmax = max(crop_gt.max(), crop_computed.max())
axes[0].imshow(crop_gt, cmap='viridis', vmin=0, vmax=vmax)
axes[0].set_title('Ground Truth Magnitude [px]')
axes[1].imshow(crop_computed, cmap='viridis', vmin=0, vmax=vmax)
axes[1].set_title('Computed Magnitude [px]')
err = crop_computed - crop_gt
lim = np.abs(err).max()
im = axes[2].imshow(err, cmap='seismic', vmin=-lim, vmax=lim)
axes[2].set_title(f'Error [px]  (RMSE={rmse:.3f})')
plt.colorbar(im, ax=axes[2], fraction=0.046)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'accuracy_comparison.png', dpi=150)
plt.show()

---
## Summary

| Step | Status |
|------|--------|
| Synthetic data generation (Cartesian + Axisymmetric) | ✅ |
| Gaussian pre-filtering | ✅ |
| Cross-correlation displacement | ✅ |
| Farneback optical flow | ✅ (if opencv-python installed) |
| Abel inversion | ✅ (if PyAbel installed) |
| Publication-quality figures | ✅ |
| HDF5 / NPY export | ✅ |
| JSON processing log | ✅ |

Next steps with real data:
- Point `--input` at your `.mraw` or TIFF folder
- Adjust `config_example.yaml` with your camera parameters
- Run `bos_process --config config.yaml`